# 02 — Train và audit base model (chỉ base classes)

**Input:** output NB1. **GPU:** cần (T4/P100). **Output:** `art/base/`.

In [ ]:
# === Setup (giống nhau ở mọi notebook) ===
SMOKE = True   # True: pipeline thật trên tập nhỏ. Run thật: False
REPO_URL = "https://github.com/Le-Anh-Duy/topicx-yolo-expansion.git"
COMMIT = "main"                                    # run thật: pin commit hash
import os, subprocess, sys
R = "/tmp/repo"
if not os.path.exists(R):
    subprocess.run(["git", "clone", "-q", REPO_URL, R], check=True)
subprocess.run(["git", "-C", R, "fetch", "-q"], check=True)
subprocess.run(["git", "-C", R, "checkout", "-q", COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{R}/requirements.txt"], check=True)
sys.path.insert(0, f"{R}/src")
import numpy as np, pandas as pd
from topicx import common as C
cfg = C.load_config(SMOKE)
print(cfg["art"])

In [ ]:
STAGE = "base"
# Test protocol (CPU, vài giây). Lỗi thì dừng notebook.
subprocess.run([sys.executable, "-m", "pytest", "-q", f"{R}/tests"], check=True)
C.env_report(cfg, STAGE)

In [ ]:
from topicx import data as D, yolo as Y
meta = C.load(C.find(cfg, "splits/meta.json"))
base_names = meta["base_names"]
paths = D.split_paths(cfg)
pub_boxes = pd.read_csv(C.find(cfg, "splits/public_boxes.csv"))
ids = {s: C.read_ids(C.find(cfg, f"splits/{s}_ids.txt")) for s in ("base", "dev")}
assert not set(pub_boxes.image[pub_boxes.cls == meta["novel"]]) & set(ids["base"]), "base train có novel"
YD = C.SCRATCH / "yolo"
tr = D.write_yolo(YD / "base_train", ids["base"], pub_boxes, base_names, paths, cfg["img_wh"])
dv = D.write_yolo(YD / "dev_base", ids["dev"], pub_boxes, base_names, paths, cfg["img_wh"])
yml = D.write_data_yaml(YD / "base.yaml", base_names, tr, dv)

In [ ]:
run = C.WORK / cfg["art"] / "base" / "run"
res = Y.train(cfg["yolo"]["init"], yml, run, cfg["yolo"]["base"], seed=cfg["seed_split"], imgsz=cfg["yolo"]["imgsz"])
res

In [ ]:
# Audit
from ultralytics import YOLO
W = run / f'{cfg["yolo"]["weights"]}.pt'
assert list(YOLO(str(W)).names.values()) == base_names
ap = Y.eval_ap(W, yml, "val", cfg["yolo"]["imgsz"], cfg["eval"]["batch"])
display(pd.DataFrame(ap).T)
secs = float((run / "DONE").read_text())
per = secs / (len(ids["base"]) * cfg["yolo"]["base"]["epochs"])
C.dump({"seconds": secs, "sec_per_img_epoch": per, "dev_ap": ap, "weights": W.name}, C.out(cfg, "base/meta.json"))
n_runs = len(cfg["seeds"]) * len(cfg["ks"]) * len(cfg["branches"] + cfg["analysis_branches"])
est = sum(per * (cfg["replay"] + k) * cfg["yolo"]["finetune"]["epochs"] for k in cfg["ks"]) * n_runs / len(cfg["ks"])
print(f"base: {secs/3600:.2f} h; ước tính NB4 ({n_runs} run, chưa tính overhead): {est/3600:.1f} h GPU")